## DPO (Direct Preference Optimization) with Phi-3 Mini — QLoRA on Google Colab

This notebook builds on top of the Basic QLoRA notebook. If you haven't gone through that one yet, do it first — the concepts of quantization, LoRA adapters, and SFTTrainer are prerequisites for what follows.

**What we're doing here:**

In the Basic QLoRA notebook, we did **SFT (Supervised Fine-Tuning)** — we taught the model *what to say* by showing it instruction-response pairs (Yoda sentences). Now, we take the next step in the post-training pipeline:

```
Base Model → SFT (already done by Microsoft) → DPO (what we're doing here)
```

**DPO teaches the model *which response is better*.** Instead of one correct response, we give the model two responses for the same prompt — one "chosen" (good) and one "rejected" (bad). The model learns to prefer the chosen style over the rejected one.

**Memory challenge:** DPO normally needs **two models** in memory — a policy model (being trained) and a reference model (frozen copy for KL penalty). On a T4 with 16GB, that's too tight. Our trick: load one model in 4-bit, add a LoRA adapter. During training, the adapter = policy model; with adapter disabled = reference model. One model, two roles.

### Setup

For better reproducibility during training, use the pinned versions below, the same versions used in the Basic QLoRA notebook:

In [1]:
!pip install transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 29.3 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8

### Imports

In [2]:
import os
import torch
from contextlib import nullcontext
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

## Loading a Quantized Base Model

This is the same setup as the Basic QLoRA notebook — we load Phi-3 Mini in 4-bit using `BitsAndBytesConfig`.

**Why do we use the Instruct model?** DPO expects a model that already knows how to follow instructions. Microsoft already did SFT to create the Instruct variant. DPO refines *which style of response* the model should prefer — so it must already be capable of *producing* responses in the first place.

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)

repo_id = "microsoft/Phi-3-mini-4k-instruct"

model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="cuda:0",
    quantization_config=bnb_config,
)

# IMPORTANT: disable KV cache during training — it wastes memory
# (KV cache is for fast inference, not training)
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [4]:
print(f"Memory footprint: {model.get_memory_footprint()/1e6:.1f} MB")

Memory footprint: 2206.3 MB


## Setting Up Low-Rank Adapters (LoRA)

Same three steps as the Basic QLoRA notebook:
1. Call `prepare_model_for_kbit_training()` for numerical stability
2. Create a `LoraConfig`
3. But **DON'T** call `get_peft_model()` — more on this below

The `target_modules` are the same Phi-3 layers from the Basic notebook: `qkv_proj`, `o_proj`, `gate_up_proj`, `down_proj`.

In [5]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
)

**Why no `get_peft_model()` here?**

In the Basic QLoRA notebook, we called `get_peft_model(model, config)` to attach the LoRA adapters ourselves, then passed the model to `SFTTrainer`.

For DPO, we do it differently — we pass the **bare model** and the `lora_config` separately to `DPOTrainer`. The trainer attaches the adapters internally. This is necessary because DPO needs to control when the adapter is **active** (policy model) vs **disabled** (reference model) during training.

If we attached the adapters ourselves, the trainer wouldn't be able to cleanly toggle them on and off.

## Loading the Tokenizer

### The PAD Token — Same Trick as the Basic Notebook

Remember the issue from the Basic QLoRA notebook? In Phi-3, the EOS token and the default PAD token are the same (`<|endoftext|>`). This caused the trainer to mask EOS tokens in the labels, so the model never learned when to stop generating.

**The fix (same as before):** assign the UNK token as PAD token, keeping EOS unique and unmaskable.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

print(f"EOS token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")
print(f"UNK token: {tokenizer.unk_token} (id: {tokenizer.unk_token_id})")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

EOS token: <|endoftext|> (id: 32000)
PAD token: <unk> (id: 0)
UNK token: <unk> (id: 0)


### Phi-3 Chat Template

Let's see how Phi-3 formats a conversation. Remember from the Basic notebook — Phi-3 uses `<|user|>`, `<|assistant|>`, and `<|end|>` tags:

In [7]:
sample_messages = [
    {"role": "user", "content": "What is gravity?"},
    {"role": "assistant", "content": "Gravity is a fundamental force that attracts objects toward each other."},
]

print(tokenizer.apply_chat_template(sample_messages, tokenize=False))

<|user|>
What is gravity?<|end|>
<|assistant|>
Gravity is a fundamental force that attracts objects toward each other.<|end|>
<|endoftext|>


## The Preference Dataset — This Is What Makes DPO Different

### SFT vs DPO Data Format

In the Basic QLoRA notebook, our SFT data looked like this — **one correct response** per prompt:

```python
# SFT format (what we used in the Basic notebook)
{"messages": [
    {"role": "user", "content": "The Force is strong in you!"},
    {"role": "assistant", "content": "Strong in you, the Force is! Hrrrmm."}
]}
```

For DPO, we need **two responses** per prompt — one chosen (good) and one rejected (bad). The `trl-lib/ultrafeedback_binarized` dataset uses an **implicit prompt** format — each of `chosen` and `rejected` contains the **full conversation** (user message + assistant response):

```python
# DPO format — implicit prompt (the prompt is inside chosen/rejected)
{
    "chosen": [
        {"role": "user", "content": "How do I learn Python?"},
        {"role": "assistant", "content": "Start with basics like variables and loops..."}
    ],
    "rejected": [
        {"role": "user", "content": "How do I learn Python?"},
        {"role": "assistant", "content": "Just Google it, it's not that hard."}
    ]
}
```

Notice: there's **no separate `prompt` column**. The user message (prompt) is the first element inside both `chosen` and `rejected`. The `DPOTrainer` automatically extracts the prompt from these columns — you don't need to do it manually.

### Dataset: UltraFeedback Binarized

We use `trl-lib/ultrafeedback_binarized` — the **official TRL example dataset** for DPO. It contains prompts with chosen/rejected response pairs where preferences were judged by GPT-4.

Why this dataset?
- Already in **conversational format** (list of message dicts) — DPOTrainer auto-applies Phi-3's chat template
- Uses **implicit prompt** format — DPOTrainer extracts the prompt automatically
- Chosen/rejected differences are **clearly visible** — you can see *why* one response is preferred
- **Official TRL dataset** — zero compatibility issues with DPOTrainer

In [8]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset size: 62135 examples
Columns: ['chosen', 'rejected', 'score_chosen', 'score_rejected']


### Inspecting the Data — What Does a Preference Pair Look Like?

Let's look at one example to understand the structure. Pay attention to **why** the chosen response is preferred over the rejected one:

In [10]:
example = dataset[0]

print("Columns in dataset:", dataset.column_names)
print()

# The dataset uses "implicit prompt" format:
# chosen = [user_message, assistant_response]  (the good response)
# rejected = [user_message, assistant_response]  (the bad response)
# The prompt (user message) is inside chosen[0] and rejected[0]

print("=" * 70)
print("PROMPT (embedded in chosen[0] — the user's question):")
print("=" * 70)
print(f"  [{example['chosen'][0]['role']}]: {example['chosen'][0]['content'][:300]}")

print()
print("=" * 70)
print("CHOSEN response (the GOOD one):")
print("=" * 70)
print(f"  [{example['chosen'][1]['role']}]: {example['chosen'][1]['content'][:300]}")

print()
print("=" * 70)
print("REJECTED response (the BAD one):")
print("=" * 70)
print(f"  [{example['rejected'][1]['role']}]: {example['rejected'][1]['content'][:300]}")

Columns in dataset: ['chosen', 'rejected', 'score_chosen', 'score_rejected']

PROMPT (embedded in chosen[0] — the user's question):
  [user]: Use the pygame library to write a version of the classic game Snake, with a unique twist

CHOSEN response (the GOOD one):
  [assistant]: Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:

1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.
2. Next, we'll create the game'

REJECTED response (the BAD one):
  [assistant]: Sure, here's an example of how to write a version of Snake game with a unique twist using the Pygame library:
```python
import pygame

class SnakeGame:
    def __init__(self, game_width, game_height):
        pygame.init()
        screen = pygame.display.set_mode((game_width, game_height))
        p


### Subsampling for Colab Free Tier

The full dataset has ~61,000 examples. On a T4 with QLoRA, training on all of it would take many hours and you'd hit Colab session timeouts. We subsample to **1,000 examples** for a demo run.

In production, you'd use the full dataset on better hardware.

In [11]:
dataset = dataset.shuffle(seed=42).select(range(1000))

# Train/eval split
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train: {len(train_dataset)} examples")
print(f"Eval:  {len(eval_dataset)} examples")

Train: 900 examples
Eval:  100 examples


## DPO Training Configuration

### Comparing SFTConfig vs DPOConfig

Most parameters are familiar from the Basic QLoRA notebook. The DPO-specific ones are new:

| Parameter | SFTConfig (Basic Notebook) | DPOConfig (This Notebook) | Why Different? |
|-----------|---------------------------|--------------------------|----------------|
| `beta` | N/A | `0.1` | Controls how much model can deviate from reference. 0.1 is the DPO paper's default. |
| `loss_type` | N/A | `"sigmoid"` | The original DPO loss function. |
| `max_length` | `64` (short Yoda sentences) | `1024` (full prompt + response) | DPO processes much longer sequences — prompt + chosen + rejected responses |
| `max_prompt_length` | N/A | `512` | Max tokens for just the prompt part |
| `packing` | `True` | N/A | DPO doesn't support packing — each sample is a distinct preference pair |
| `per_device_train_batch_size` | `16` | `1` | DPO sequences are much longer → more memory per sample |

### What is beta?

`beta` is **THE** core DPO hyperparameter. It controls the KL penalty — how much the trained model is allowed to deviate from the reference model.

- **Low beta (0.05):** model changes a lot → more aligned but risks losing general capabilities
- **High beta (0.5):** model stays close to reference → safer but less aligned
- **0.1:** the standard default from the DPO paper

In [12]:
dpo_config = DPOConfig(
    ## DPO-SPECIFIC parameters (NEW compared to SFTConfig)
    beta=0.1,                # KL penalty — THE core DPO hyperparameter
    loss_type="sigmoid",     # original DPO loss from the paper
    max_length=1024,         # max total tokens (prompt + response)
    max_prompt_length=512,   # max prompt tokens

    ## Memory usage (same philosophy as Basic QLoRA notebook)
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=4,     # effective batch size = 1 * 4 = 4
    per_device_train_batch_size=1,     # T4 memory constraint — DPO is heavier than SFT
    per_device_eval_batch_size=1,
    auto_find_batch_size=True,

    ## Training parameters
    num_train_epochs=1,        # 1 epoch for demo; use 2-3 for real training
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",

    ## Logging parameters
    logging_steps=10,
    logging_dir="./logs",
    output_dir="./phi3-mini-dpo-adapter",
    report_to="none",
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    seed=42,

    bf16=torch.cuda.is_bf16_supported(including_emulation=False),
)

## DPO Trainer — The Key Difference from SFTTrainer

### How the Reference Model Works (The LoRA Trick)

In the Basic QLoRA notebook, `SFTTrainer` just trained the model. Simple.

`DPOTrainer` is more complex because DPO needs **two models**:
1. **Policy model** (being trained) — "what should the model say?"
2. **Reference model** (frozen) — "what would the model say without DPO?" (used for KL penalty)

Loading two 3.8B models would blow up T4 memory. Here's the trick:

When we pass `peft_config` and set `ref_model=None`:
- `DPOTrainer` adds a LoRA adapter to the base model
- **Policy inference:** LoRA adapter is **enabled** → base model + adapter = policy
- **Reference inference:** LoRA adapter is **disabled** → just the base model = reference

**One model in memory, serving both roles.** This is what makes DPO feasible on Colab.

Compare this with the Basic notebook where we just did:
```python
trainer = SFTTrainer(model=model, ...)
```

Now we have two extra arguments — `ref_model=None` and `peft_config`:

In [13]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,             # ← use base model (LoRA disabled) as reference
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,    # ← DPOTrainer attaches LoRA internally
)

Extracting prompt in train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
trainable_parms = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot_parms = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters:             {trainable_parms/1e6:.2f}M")
print(f"Total parameters:                 {tot_parms/1e6:.2f}M")
print(f"Fraction of trainable parameters: {100*trainable_parms/tot_parms:.2f}%")

Trainable parameters:             12.58M
Total parameters:                 2021.72M
Fraction of trainable parameters: 0.62%


## Training — What to Watch

### DPO Metrics (Different from SFT!)

In the Basic QLoRA notebook, we watched the **training loss** go down. That was the only metric that mattered.

DPO logs **reward metrics** that tell you whether the model is actually learning to prefer chosen over rejected:

| Metric | What it means | Good direction |
|--------|--------------|----------------|
| `rewards/chosen` | Average reward for chosen responses | Should go **UP** |
| `rewards/rejected` | Average reward for rejected responses | Should go **DOWN** |
| `rewards/margins` | `chosen_reward - rejected_reward` | Should **INCREASE** |
| `rewards/accuracies` | How often `chosen_reward > rejected_reward` | Should approach **1.0** |
| `loss` | DPO loss | Should **decrease** |

**The most important metric is `rewards/margins`** — if this increases, the model is successfully learning to differentiate between good and bad responses.

<blockquote class="tip">
  <p>
    If <code>rewards/accuracies</code> starts near 0.5, it means the model can't tell chosen from rejected (random guessing). By the end of training, it should be close to 1.0.
  </p>
</blockquote>

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
50,0.620700,0.685437,-0.197995,-0.279560,0.580000,0.081566,-336.483704,-309.669617,19.799669,19.780184
100,0.558400,0.630823,-0.352150,-0.696886,0.680000,0.344736,-338.025269,-313.842865,19.723450,19.661541


## Saving the Adapter

Just like in the Basic QLoRA notebook — we save the small LoRA adapter, not the full model:

In [ ]:
trainer.save_model("local-phi3-mini-dpo-adapter")
tokenizer.save_pretrained("local-phi3-mini-dpo-adapter")

print("Saved files:")
for f in os.listdir("local-phi3-mini-dpo-adapter"):
    print(f"  {f}")

## Querying the DPO-Aligned Model

Same helper functions from the Basic QLoRA notebook. We build a prompt using `apply_chat_template`, feed it to the model, and generate a response.

The questions below are deliberately about topics where DPO alignment should shine — making the model more helpful, empathetic, and thoughtful compared to a plain base model.

In [ ]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [
        {"role": "user", "content": sentence},
    ]
    prompt = tokenizer.apply_chat_template(
        converted_sample,
        tokenize=False,
        add_generation_prompt=True,
    )
    return prompt


def generate(model, tokenizer, prompt, max_new_tokens=256, skip_special_tokens=False):
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    ctx = (
        torch.autocast(device_type=model.device.type, dtype=model.dtype)
        if model.dtype in [torch.float16, torch.bfloat16]
        else nullcontext()
    )
    with ctx:
        generation_output = model.generate(
            **tokenized_input,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    output = tokenizer.batch_decode(
        generation_output, skip_special_tokens=skip_special_tokens
    )
    return output[0]

In [ ]:
sentence = "I'm feeling really overwhelmed with work and life. What should I do?"
prompt = gen_prompt(tokenizer, sentence)
print("PROMPT:")
print(prompt)
print()
print("=" * 70)
print("DPO-ALIGNED RESPONSE:")
print("=" * 70)
print(generate(model, tokenizer, prompt))

In [ ]:
# Try another prompt
sentence2 = "Explain quantum computing to a 10-year-old."
prompt2 = gen_prompt(tokenizer, sentence2)
print("DPO-ALIGNED RESPONSE:")
print("=" * 70)
print(generate(model, tokenizer, prompt2))

### Pushing to the Hub (Optional)

If you'd like to share your adapter with everyone, log in and push:

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# trainer.push_to_hub()

## Summary: What Changed from the Basic QLoRA Notebook

| Aspect | Basic QLoRA (SFT) | This Notebook (DPO) |
|--------|-------------------|---------------------|
| **Trainer** | `SFTTrainer` | `DPOTrainer` |
| **Config** | `SFTConfig` | `DPOConfig` |
| **Model** | `microsoft/Phi-3-mini-4k-instruct` | Same model — `microsoft/Phi-3-mini-4k-instruct` |
| **Dataset format** | Single response: `messages` (user → assistant) | Preference pair: `prompt`, `chosen`, `rejected` |
| **Dataset** | `dvgodoy/yoda_sentences` (720 examples) | `trl-lib/ultrafeedback_binarized` (1000 subsampled) |
| **Loss function** | Cross-entropy on the response tokens | Log-probability gap between chosen & rejected |
| **What it teaches** | *What to say* (memorize this response) | *Which response is better* (comparison) |
| **LoRA attachment** | `get_peft_model(model, config)` — we do it | `peft_config` passed to trainer — trainer does it |
| **Reference model** | Not needed | Needed (handled via LoRA enable/disable trick) |
| **Key metric** | Training loss ↓ | Reward margins ↑, Accuracy → 1.0 |
| **Key hyperparameter** | `learning_rate`, `num_train_epochs` | `beta` (KL penalty coefficient) |
| **Batch size** | 16 (short Yoda sentences) | 1 (long preference pairs) |
| **Sequence length** | 64 tokens | 1024 tokens |

The core code structure is almost identical — same imports, same quantization, same LoRA layers. **The data format and the trainer class are the two things that actually change.** Everything else is the same QLoRA machinery you already know from the Basic notebook.

That's a wrap! You've now aligned an LLM with human preferences using DPO. 🎉